# PyTorch Experiment: MLP Mass Regression

This notebook mirrors the first sklearn baseline with a simple feed-forward neural network so model complexity can increase without changing the dataset framing.

In [2]:
from pathlib import Path

import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path.cwd() / "data"
eu_df = pd.read_csv(
    DATA_DIR / "exoplanet.eu_catalog_28-05-26_22_04_09.csv", low_memory=False
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [3]:
target = "mass"
feature_columns = [
    "mass_sini",
    "radius",
    "orbital_period",
    "semi_major_axis",
    "eccentricity",
    "inclination",
    "temp_calculated",
    "star_distance",
    "star_metallicity",
    "star_mass",
    "star_radius",
    "star_age",
    "star_teff",
    "planet_status",
    "detection_type",
    "mass_measurement_type",
    "star_sp_type",
]

model_df = eu_df[[target, *feature_columns]].copy()
model_df = model_df[model_df[target].notna()]

X = pd.get_dummies(model_df.drop(columns=[target]), dummy_na=True)
X = X.fillna(X.median(numeric_only=True)).fillna(0.0)
y = model_df[target].astype("float32")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [5]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32, device=device)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32, device=device)
y_train_tensor = torch.tensor(
    y_train.to_numpy(), dtype=torch.float32, device=device
).view(-1, 1)
y_test_tensor = torch.tensor(
    y_test.to_numpy(), dtype=torch.float32, device=device
).view(-1, 1)


class Regressor(torch.nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = torch.nn.Sequential(
            torch.nn.Linear(input_dim, 128),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.2),
            torch.nn.Linear(128, 64),
            torch.nn.ReLU(),
            torch.nn.Linear(64, 1),
        )

    def forward(self, inputs):
        return self.network(inputs)


model = Regressor(X_train_tensor.shape[1]).to(device)
display(model)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
display(optimizer)
loss_fn = torch.nn.MSELoss()
display(loss_fn)

Regressor(
  (network): Sequential(
    (0): Linear(in_features=298, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Linear(in_features=64, out_features=1, bias=True)
  )
)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)

MSELoss()

In [6]:
loss_history = []

for epoch in range(50):
    model.train()
    optimizer.zero_grad()
    predictions = model(X_train_tensor)
    loss = loss_fn(predictions, y_train_tensor)
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())

loss_history[-5:]

[3.4990217685699463,
 3.5032382011413574,
 3.4368560314178467,
 3.4251134395599365,
 3.3813607692718506]

In [7]:
model.eval()
with torch.no_grad():
    test_predictions = model(X_test_tensor).cpu().numpy().ravel()

results = pd.DataFrame(
    {
        "actual_mass": y_test.reset_index(drop=True),
        "predicted_mass": test_predictions,
    }
)
results.head()

,actual_mass,predicted_mass
0,2.54000,2.043934
1,0.02454,0.350835
2,0.06010,-0.002221
3,0.00135,0.642244
4,0.04273,-0.049664
